
# Investigating a data-driven stability threshold

This notebook tests whether using **15th place** as the minimum threshold for Premier League stability is defensible from the data.

The key idea:

> Instead of choosing 15th arbitrarily, examine how a club's current finishing position relates to its probability of remaining stable over the following seasons.

Expected input:

`data/master/modelling_features.csv`

or equivalent master dataset containing at least:

- `team_name`
- `year`
- `position`
- `next_position`
- `position_t_plus_2`
- `survived_next_season`
- `survived_next_2_seasons`
- `top_15_next_season`
- `top_15_next_2_seasons`
- `big_six`


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/master/modelling_features.csv")
OUTPUT_DIR = Path("../data/eda/threshold_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()



## 1. Check available seasons and remove rows without future outcome data

For a two-season future stability target, the most recent two seasons usually cannot be evaluated yet.  
Those rows should not be included when calculating historical future outcomes.


In [ ]:

required_cols = [
    "team_name",
    "year",
    "position",
    "big_six",
    "survived_next_season",
    "survived_next_2_seasons",
    "top_15_next_season",
    "top_15_next_2_seasons",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Years available:", sorted(df["year"].dropna().unique()))
print("Rows by year:")
print(df.groupby("year").size())



## 2. Primary sample: non-Big-Six clubs only

This aligns the threshold analysis to Sunderland's likely peer group rather than letting the Premier League elite dominate the result.


In [ ]:

analysis_df = df[df["big_six"] == 0].copy()

# Keep only seasons where a two-year outcome is observable.
# If you have explicit target columns already calculated, this is safest:
analysis_df = analysis_df[analysis_df["year"] <= analysis_df["year"].max() - 2].copy()

print(analysis_df.shape)
print(analysis_df.groupby("year").size())



## 3. Stability probability by current finishing position

This is the key analysis for justifying a threshold.

If clubs finishing 15th or above show materially higher future stability than clubs finishing 16th/17th, then 15th becomes a defensible cutoff.


In [ ]:

position_summary = (
    analysis_df
    .groupby("position")
    .agg(
        club_seasons=("team_name", "count"),
        survived_next_season_rate=("survived_next_season", "mean"),
        survived_next_2_seasons_rate=("survived_next_2_seasons", "mean"),
        top_15_next_season_rate=("top_15_next_season", "mean"),
        top_15_next_2_seasons_rate=("top_15_next_2_seasons", "mean"),
    )
    .reset_index()
)

position_summary.to_csv(OUTPUT_DIR / "position_stability_summary.csv", index=False)

position_summary


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    position_summary["position"],
    position_summary["top_15_next_2_seasons_rate"],
    marker="o",
    linewidth=2,
)

ax.axvline(15, linestyle="--", linewidth=1)
ax.text(
    15.2,
    position_summary["top_15_next_2_seasons_rate"].max() * 0.9,
    "15th-place threshold",
    va="center",
)

ax.set_title("Future stability probability by current finishing position")
ax.set_xlabel("Current Premier League finishing position")
ax.set_ylabel("Probability of top-15 finish in each of next two seasons")
ax.set_xticks(range(1, 21))
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "position_vs_future_stability.png", dpi=300)
plt.show()



## 4. Compare grouped finishing bands

This makes the finding easier to communicate in a deck.

Suggested bands:

- **Top half:** 1st–10th
- **Sustainable zone:** 11th–15th
- **At-risk survival:** 16th–17th
- **Relegated:** 18th–20th

For your deck, the key comparison is usually:

> 11th–15th vs 16th–17th


In [ ]:

def position_band(pos):
    if pos <= 10:
        return "1st-10th: Top half / European-adjacent"
    if pos <= 15:
        return "11th-15th: Sustainable zone"
    if pos <= 17:
        return "16th-17th: At-risk survival"
    return "18th-20th: Relegated"

analysis_df["position_band"] = analysis_df["position"].apply(position_band)

band_summary = (
    analysis_df
    .groupby("position_band")
    .agg(
        club_seasons=("team_name", "count"),
        avg_current_position=("position", "mean"),
        survived_next_season_rate=("survived_next_season", "mean"),
        survived_next_2_seasons_rate=("survived_next_2_seasons", "mean"),
        top_15_next_season_rate=("top_15_next_season", "mean"),
        top_15_next_2_seasons_rate=("top_15_next_2_seasons", "mean"),
    )
    .reset_index()
)

band_order = [
    "1st-10th: Top half / European-adjacent",
    "11th-15th: Sustainable zone",
    "16th-17th: At-risk survival",
    "18th-20th: Relegated",
]

band_summary["position_band"] = pd.Categorical(
    band_summary["position_band"],
    categories=band_order,
    ordered=True,
)

band_summary = band_summary.sort_values("position_band")
band_summary.to_csv(OUTPUT_DIR / "position_band_stability_summary.csv", index=False)

band_summary


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

plot_df = band_summary.copy()

ax.bar(
    plot_df["position_band"].astype(str),
    plot_df["top_15_next_2_seasons_rate"],
)

ax.set_title("Future stability by current finishing band")
ax.set_ylabel("Probability of top-15 finish in each of next two seasons")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)

plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "position_band_future_stability.png", dpi=300)
plt.show()



## 5. Test alternative thresholds

This checks whether 15th is a sensible choice compared with other cutoffs.

For each threshold, calculate the future stability rate for clubs finishing at or above that position.


In [ ]:

threshold_rows = []

for threshold in range(10, 18):
    group = analysis_df[analysis_df["position"] <= threshold]
    comparison = analysis_df[analysis_df["position"] > threshold]

    threshold_rows.append(
        {
            "threshold_position_or_better": threshold,
            "included_club_seasons": len(group),
            "excluded_club_seasons": len(comparison),
            "included_top_15_next_2_seasons_rate": group["top_15_next_2_seasons"].mean(),
            "excluded_top_15_next_2_seasons_rate": comparison["top_15_next_2_seasons"].mean(),
            "gap": (
                group["top_15_next_2_seasons"].mean()
                - comparison["top_15_next_2_seasons"].mean()
            ),
        }
    )

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.to_csv(OUTPUT_DIR / "threshold_sensitivity_summary.csv", index=False)

threshold_summary


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    threshold_summary["threshold_position_or_better"],
    threshold_summary["gap"],
    marker="o",
    linewidth=2,
)

ax.axvline(15, linestyle="--", linewidth=1)
ax.text(
    15.15,
    threshold_summary["gap"].max() * 0.9,
    "15th",
    va="center",
)

ax.set_title("Threshold sensitivity: separation in future stability")
ax.set_xlabel("Current position threshold")
ax.set_ylabel("Stability rate gap: included vs excluded")
ax.set_xticks(range(10, 18))
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "threshold_sensitivity_gap.png", dpi=300)
plt.show()



## 6. Suggested deck language

Use this wording if the analysis supports the 15th-place cutoff:

> Rather than selecting 15th place arbitrarily, I tested how current finishing position related to future Premier League stability. Among non-Big-Six clubs, the probability of remaining a top-15 side over the following two seasons declined materially below the 15th-place band. I therefore used 15th as a practical threshold for established Premier League status, while treating 16th–17th as an elevated-risk survival zone.

If the result is less clear, use:

> The threshold should be treated as a pragmatic benchmark rather than a hard rule. The purpose is to separate clubs operating outside the relegation battle from those surviving closer to the bottom three.
